# Hello Backtest: Simple SMA Crossover Strategy

This notebook demonstrates:
- Loading OHLCV data with `CsvDataLoader`
- Implementing a simple SMA crossover strategy
- Running a backtest with `VectorBTEngine`
- Generating risk analytics and QuantStats tearsheet

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path if running from notebooks/
if Path.cwd().name == "notebooks":
    sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from medusa.analytics.risk import RiskAnalytics
from medusa.backtest.vectorbt_engine import VectorBTEngine
from medusa.strategies.examples.sma_crossover import SMACrossoverStrategy

# Optional: QuantStats tearsheet
try:
    import quantstats as qs
    HAS_QS = True
except ImportError:
    HAS_QS = False
    print("QuantStats not installed. Tearsheet will be skipped.")

## 1. Generate Synthetic OHLCV Data

Since we don't have a CSV file yet, we'll generate synthetic data with a trend.

In [ ]:
# Generate synthetic price data with upward trend
np.random.seed(42)
dates = pd.date_range("2023-01-01", periods=500, freq="1h")

# Geometric Brownian Motion with drift
mu = 0.0001  # Small positive drift
sigma = 0.02  # Volatility
returns = np.random.normal(mu, sigma, len(dates))
price = 100 * np.exp(np.cumsum(returns))

# Generate OHLC from close prices
high = price * (1 + np.abs(np.random.normal(0, 0.005, len(dates))))
low = price * (1 - np.abs(np.random.normal(0, 0.005, len(dates))))
open_price = price + np.random.normal(0, 0.2, len(dates))
volume = np.random.randint(1000, 10000, len(dates))

df = pd.DataFrame(
    {
        "open": open_price,
        "high": high,
        "low": low,
        "close": price,
        "volume": volume,
    },
    index=dates,
)

print(f"Generated {len(df)} bars of synthetic data")
df.head()

In [ ]:
# Plot price
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["close"], label="Close Price")
plt.title("Synthetic Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

## 2. Implement SMA Crossover Strategy

Buy when fast SMA crosses above slow SMA, sell when it crosses below.

In [ ]:
# Create strategy instance
strategy = SMACrossoverStrategy(fast_period=10, slow_period=30)

# Generate signals
entries, exits = strategy.generate_signals(df)

print(f"Entry signals: {entries.sum()}")
print(f"Exit signals: {exits.sum()}")

In [ ]:
# Visualize signals
from medusa.features.technical import sma

fast_sma = sma(df["close"], 10)
slow_sma = sma(df["close"], 30)

plt.figure(figsize=(14, 7))
plt.plot(df.index, df["close"], label="Close Price", alpha=0.7)
plt.plot(df.index, fast_sma, label="Fast SMA (10)", linewidth=1.5)
plt.plot(df.index, slow_sma, label="Slow SMA (30)", linewidth=1.5)

# Mark entry/exit points
plt.scatter(df.index[entries], df.loc[entries, "close"], color="green", marker="^", s=100, label="Buy", zorder=5)
plt.scatter(df.index[exits], df.loc[exits, "close"], color="red", marker="v", s=100, label="Sell", zorder=5)

plt.title("SMA Crossover Signals")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

## 3. Run Backtest with VectorBT

Execute the strategy and analyze performance.

In [ ]:
# Initialize backtesting engine
engine = VectorBTEngine(
    initial_capital=100_000,
    commission=0.001,
    slippage=0.0005,
)

# Run backtest
portfolio = engine.run_signals(
    price=df["close"],
    entries=entries,
    exits=exits,
)

In [ ]:
# Display portfolio statistics
print("\n" + "="*60)
print("BACKTEST RESULTS")
print("="*60)
print(f"Total Return: {portfolio.total_return():.2%}")
print(f"Sharpe Ratio: {portfolio.sharpe_ratio():.2f}")
print(f"Sortino Ratio: {portfolio.sortino_ratio():.2f}")
print(f"Max Drawdown: {portfolio.max_drawdown():.2%}")
print(f"Win Rate: {portfolio.trades.win_rate:.2%}")
print(f"Total Trades: {portfolio.trades.count}")
print("="*60)

## 4. Generate Risk Analytics Report

In [ ]:
# Extract returns
returns = portfolio.returns()

# Generate full risk report
risk_report = RiskAnalytics.full_report(returns)

# Display as DataFrame
risk_df = pd.DataFrame.from_dict(risk_report, orient="index", columns=["Value"])
print("\nRISK ANALYTICS")
print(risk_df)

## 5. QuantStats Tearsheet (Optional)

Generate a comprehensive tearsheet with all performance metrics.

In [ ]:
if HAS_QS:
    qs.extend_pandas()

    # Generate tearsheet
    qs.reports.html(
        returns,
        output="tearsheet.html",
        title="SMA Crossover Strategy",
    )

    print("Tearsheet saved to tearsheet.html")

    # Display metrics
    qs.reports.metrics(returns, mode="full")
else:
    print("Install quantstats to generate tearsheet: pip install quantstats")

## 6. Plot Equity Curve

In [ ]:
# Plot cumulative returns
cumulative = portfolio.cumulative_returns()

plt.figure(figsize=(12, 6))
plt.plot(cumulative.index, cumulative.values, label="Strategy")
plt.axhline(y=1.0, color="gray", linestyle="--", label="Break-even")
plt.title("Cumulative Returns: SMA Crossover Strategy")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True)
plt.show()

## Next Steps

- Try different SMA periods (e.g., 20/50, 50/200)
- Add stop-loss / take-profit rules
- Optimize parameters with `VectorBTEngine.optimize_params()`
- Load real data from CSV with `CsvDataLoader.load_ohlcv()`